# Reading Pharmacy Claims with an AI Assistant (Claude Haiku)

**Audience:** healthcare professionals learning Python. No prior AI experience needed.

Pharmacies bill insurance using the **NCPDP** standard — dense lines of codes that are
hard for humans to read. In this notebook you will use **Claude Haiku** (a fast, low-cost
AI model, used live through its API) to translate those cryptic claim lines into plain English.

**What you will learn**

1. How to install and set up the `anthropic` Python library
2. How to make your first API call to an AI model
3. How to wrap that call in a reusable Python function
4. How to apply it to a real (synthetic) pharmacy claims file

**Before you start** you need an API key from [console.anthropic.com](https://console.anthropic.com/)
(Settings → API keys → Create key).

> ⚠️ **Important — patient privacy (HIPAA):** the file used here is 100% synthetic.
> Never send real patient data to any external API without proper agreements in place.

## Step 1 — Install the library

The `anthropic` library lets Python talk to Claude. You only need to run this once per environment.

In [1]:
%pip install anthropic --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 2 — Enter your API key

An API key is like a password that identifies you to the service.
We use `getpass` so the key is **not displayed on screen or saved in the notebook** —
never paste your key directly into code.

In [2]:
from getpass import getpass
import anthropic

api_key = getpass("Paste your Anthropic API key: ")
client = anthropic.Anthropic(api_key=api_key)
print("Client ready!")

Client ready!


## Step 3 — Load the claims file

We download the synthetic NCPDP claims file and read it line by line.
Each line is one **transaction**: the first 10 lines are claims sent by the pharmacy,
the last 10 are the insurer's responses (paid or rejected).

In [3]:
import urllib.request

SRC = "https://raw.githubusercontent.com/thousandoaks/Python4DS-II/refs/heads/main/datasets/ncpdp_pharmacy_claims.dat"
urllib.request.urlretrieve(SRC, "ncpdp_pharmacy_claims.dat")

with open("ncpdp_pharmacy_claims.dat") as f:
    lines = [line.strip() for line in f if line.strip()]

print(f"The file contains {len(lines)} transactions.")
print("\nHere is the first one:\n")
print(lines[0])

The file contains 20 transactions.

Here is the first one:

610415D0B1PHRXPLAN  1011234567890     20260701SYNTHV1   AM04C2MBR200001C1RXGRPAAM01C419680312C52AM07EM1D27000001E103D790001000101E730D530D80AM03EZ01DB1987654321AM11D94500DC125DQ5000DU4625


Unreadable, right? Codes like `AM01`, `D7`, `DQ` all have precise meanings in the NCPDP
standard, but memorizing them takes years. Let's ask an AI to do the translating.

## Step 4 — Your first API call

An API call has three key ingredients:

- **model** — which AI to use (`claude-haiku-4-5` is fast and inexpensive, ideal for learning)
- **max_tokens** — the maximum length of the answer (a token ≈ ¾ of a word)
- **messages** — what you say to the model, just like typing in a chat

The model's answer comes back in `response.content[0].text`.

In [4]:
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=500,
    messages=[
        {
            "role": "user",
            "content": "You are helping a clinician who cannot read NCPDP pharmacy "
                       "claim formats. Explain this claim line in simple plain English "
                       "(patient info, prescription, prescriber, costs):\n\n" + lines[0],
        }
    ],
)

print(response.content[0].text)

# NCPDP Claim Explanation in Plain English

## **Patient Information**
- **Member ID:** 1011234567890
- **Date of Service:** July 1, 2026

## **Prescription Details**
- **Medication:** Synthroid (levothyroxine) - a thyroid hormone medication
- **Quantity:** 90 tablets
- **Days Supply:** 90 days
- **Refills:** 0 (no refills authorized)

## **Prescriber Information**
- **NPI/Prescriber ID:** 1968031200001

## **Costs & Amounts**
- **Ingredient Cost (Drug Cost):** $790.00
- **Dispensing Fee:** $5.00
- **Insurance Pays:** $462.50
- **Patient Copay/Coinsurance:** $25.00 (this is what the patient owes)

## **In Summary**
A patient with pharmacy insurance filled a 90-day supply of Synthroid on July 1, 2026. The total cost was about $795, the insurance company paid $462.50, and the patient paid a $25 copay.


## Step 5 — Wrap it in a reusable function

Copy-pasting that block every time would get tedious. In Python we package repeated
work into a **function**. `ask_haiku` takes any question and returns the model's answer as text.

In [5]:
def ask_haiku(question):
    """Send a question to Claude Haiku and return its answer as plain text."""
    response = client.messages.create(
        model="claude-haiku-4-5",
        max_tokens=1000,
        messages=[{"role": "user", "content": question}],
    )
    return response.content[0].text


# Quick test
print(ask_haiku("In one sentence: what is the NCPDP standard used for?"))

The NCPDP standard is used for transmitting pharmacy and prescription drug information electronically between healthcare providers, pharmacies, and insurance companies.


## Step 6 — Translate several claims in a loop

A `for` loop repeats the same action for each item. Here we ask for a **one-sentence
summary** of each of the first three claims. (We keep it to three to stay fast and cheap —
each API call costs a fraction of a cent.)

In [6]:
for i, claim in enumerate(lines[:3], start=1):
    summary = ask_haiku(
        "Summarize this NCPDP pharmacy claim in ONE sentence a nurse would understand. "
        "Include date of service and amount billed:\n\n" + claim
    )
    print(f"Claim {i}: {summary}\n")

Claim 1: # Pharmacy Claim Summary

On July 1, 2026, a prescription for Synthroid (synthetic thyroid hormone) was dispensed and billed for $46.25.

Claim 2: # Pharmacy Claim Summary

On July 1, 2026, a pharmacy claim for Synthroid was submitted for $1,500.00.

Claim 3: # Pharmacy Claim Summary

On July 2, 2026, a pharmacy claim was submitted for $32.00 for a Synthroid prescription for a patient on an HMO plan.



## Step 7 — Ask questions about the whole file

We can also paste the **entire file** into a single prompt and ask questions across all
transactions at once — the way you might ask a colleague who had just reviewed the batch.

In [7]:
whole_file = "\n".join(lines)

answer = ask_haiku(
    "Below is a full NCPDP pharmacy claims file (10 claims followed by 10 insurer "
    "responses). Answer in plain English for a medical audience:\n"
    "1. How many claims were paid and how many rejected?\n"
    "2. For any rejected claim, which prescription was it and why was it rejected?\n\n"
    + whole_file
)

print(answer)

# NCPDP Claims Analysis

## Summary
- **Claims Paid: 9**
- **Claims Rejected: 1**

## Rejected Claim Details

**Claim #9 (MBR200009)** - **REJECTED**

**Reason for Rejection:** Missing or invalid prior authorization

The insurer response (AUTH0009) shows code "ANRFA1" which indicates "Authorization Not Required/Not Applicable" but with a "FB70" denial code, suggesting a prior authorization requirement was not met or documentation was missing for this prescription.

---

## Summary of Paid Claims

Claims 1-8 and 10 were successfully processed and paid:
- Claim 1 (MBR200001) - Paid
- Claim 2 (MBR200002) - Paid
- Claim 3 (MBR200003) - Paid
- Claim 4 (MBR200004) - Paid
- Claim 5 (MBR200005) - Paid
- Claim 6 (MBR200006) - Paid
- Claim 7 (MBR200007) - Paid
- Claim 8 (MBR200008) - Paid
- Claim 10 (MBR200010) - Paid


## Step 8 — Exercises (with worked solutions) 🎓

Three practical questions a pharmacy or clinical team might actually ask about this batch.
**Try writing each prompt yourself first**, then run the solution cell and compare.

Notice the pattern in every solution: it is always the same `ask_haiku()` function —
only the *question* changes. Learning to write clear questions ("prompts") is the core skill here.

### Exercise 1 — Which claim had the highest amount billed?

In NCPDP claims, the **gross amount due** (what the pharmacy bills) is the `DQ` field,
in cents with no decimal point (so `DQ7500` means $75.00).

We give Haiku that hint in the prompt — telling the model how to read the data is
half the work.

In [8]:
answer = ask_haiku(
    "Below is an NCPDP pharmacy claims file. In each claim line, the gross amount "
    "billed is the DQ field, written in cents with no decimal point (DQ7500 = $75.00).\n"
    "Which claim has the HIGHEST amount billed? Give the date of service, the "
    "prescription number (D2 field) and the amount in dollars. Answer in 2-3 "
    "sentences of plain English.\n\n" + whole_file
)

print(answer)

# Highest Amount Billed

The claim with the highest amount billed is dated **July 4, 2026**, with prescription number **D27000007**, and an amount of **$75.00**. This claim appears in the seventh line of the pharmacy claims file with a DQ field value of DQ7500.


### Exercise 2 — How many different prescribers appear in the file?

Each prescriber is identified by an **NPI** (National Provider Identifier), the 10-digit
number after `DB` in each claim. Several claims can share the same prescriber, so we
ask for *distinct* NPIs.

In [9]:
answer = ask_haiku(
    "Below is an NCPDP pharmacy claims file. The prescriber's 10-digit NPI number "
    "appears after the code DB in each claim line.\n"
    "How many DIFFERENT prescribers (distinct NPIs) appear in the file? List each "
    "distinct NPI and how many claims belong to it. Answer in plain English.\n\n"
    + whole_file
)

print(answer)

# Analysis of Prescriber NPIs in NCPDP File

After examining all claim lines in the file, I found **3 DIFFERENT prescribers**:

1. **NPI 1987654321** - 4 claims
2. **NPI 1900000032** - 3 claims
3. **NPI 1900000064** - 3 claims

**Summary:** There are 3 distinct prescriber NPIs in this pharmacy claims file, appearing after the "DB" code identifier in each claim record.


### Exercise 3 — Draft a note to the pharmacy about the rejected claim

AI models are useful for more than reading data — they can also **write for you**.
Here we ask Haiku to first find the rejection in the responses, then draft a short,
professional note the plan could send to the pharmacy.

In [10]:
note = ask_haiku(
    "Below is an NCPDP pharmacy claims file: 10 claims followed by 10 insurer "
    "responses. One response is a rejection (reject code after FB; code 70 means "
    "'Product/Service Not Covered').\n"
    "Identify the rejected claim, then draft a short, polite note (under 120 words) "
    "from the insurance plan to the pharmacy: state the prescription number, the date "
    "of service, the reason for rejection in plain language, and one suggested next "
    "step. Do not invent patient names.\n\n" + whole_file
)

print(note)

# Rejected Claim Identified

**Prescription Number:** D27000009  
**Date of Service:** July 5, 2026  
**Reject Code:** FB70 (Product/Service Not Covered)

---

## Letter to Pharmacy

Dear Pharmacy Partner,

We are writing regarding prescription D27000009, dated July 5, 2026, which we cannot process.

**Reason for Rejection:** The medication prescribed is not covered under this member's plan formulary.

**Next Step:** Please contact the member to discuss alternative covered medications, or request that the prescriber submit a prior authorization request if clinically appropriate.

We appreciate your attention to this matter.

Sincerely,  
[Insurance Plan Name]
